In [19]:

import os
import pandas as pd
import numpy as np

# file parameters
subject = 'zefir'

# set the resolution of the input images
in_plane_res_x = 10 #10 microns per pixel
in_plane_res_y = 10 #10 microns per pixel
in_plane_res_z = 50 #slice thickness of 50 microns

zfill_num = 4 #number of digits to use for zero-filling in file names, e.g. 4 means that slice the first slice will be 0000
per_slice_template = True #use a median of the slice and adjacent slices to create a slice-specific template for anchoring the registration
use_nonlin_slice_templates = False #use interpolated slices (from registrations of neighbouring 2 slices) as templates for registration, otherwise median
                                    # nonlinear slice templates take a long time and result in very jagged registrations, but may end up being useful for bring slices that are very far out of alignment back in
                                    # currently BROKEN
slice_template_type = 'median'
across_slice_smoothing_sigma = None # (None/0; pos int} sigma for smoothing across the stack (only in the slice direction), applied after stacking and before template creation
if use_nonlin_slice_templates:
    slice_template_type = [slice_template_type,'nonlin']

#this fails on server, for some reason?    
mask_zero = False #mask zeros for nighres registrations

# Control whether to use resolution information during registration
# False (default): Registration works in voxel space (ignore_res=True), better empirical performance
#                  Output images will have the specified voxel resolution in their headers, but registration itself does not use this information
# True: Registration uses physical resolution (ignore_res=False), more physically accurate
use_resolution_in_registration = True
use_signed_distance_weighting_for_registration = False #compute signed distance function from the images and use this image for registration

# scaling factor that is applied to the x and y dimensions (in-plane dimensions) to downsample the data
# rescale=5 #larger scale means that you have to change the scaling_factor, which is now done automatically just before computations
# rescale=5
rescale=40

#based on the rescale value, we adjust our in-plane resolution
#keep resolutions in microns (not mm) - only apply rescale to x and y
rescaled_in_plane_res_x = rescale * in_plane_res_x
rescaled_in_plane_res_y = rescale * in_plane_res_y
# z resolution stays at original value (not rescaled)
in_plane_res_z_microns = in_plane_res_z

actual_voxel_res = [rescaled_in_plane_res_x, rescaled_in_plane_res_y, in_plane_res_z_microns]
#if we don't want to set the voxel resolution, we can set it to None and it will be 1x1x1
voxel_res = actual_voxel_res # defines voxel resolution for output template in microns # registration itself performs much better when we do not specify the res

downsample_parallel = False #True means that we invoke Parallel, but can be much faster on HPC when set to False since it skips the Parallel overhead
max_workers = 50 #number of parallel workers to run for registration -> registration is slow but not CPU bound on an HPC (192 cores could take ??)
nonlin_interp_max_workers = 50 #number of workers to use for nonlinear slice interpolation when use_nonlin_slice_templates = True

# setup the output directory for use
if use_signed_distance_weighting_for_registration:
    sdf_tag = '_sdf'
    cost_function = 'CrossCorrelation'
    missing_slice_interp_method = 'mean'
    cortical_detail_weight = .3 #.5 equally balances cortical ribbon with inside definition of structure (.5 is equal weighting, .3 is good for stability I think!), smaller values preserve more of the interior holes but may lose some cortical detail, larger values preserve more of the cortical detail but may lose definition in the interior (since it will blend with the filled SDF)
    if rescale == 40:
        sdf_clip_val = 10
    elif rescale == 5:
        sdf_clip_val = 100
    else:
        sdf_clip_val = 50
else:
    sdf_tag = ''
    cost_function = 'MutualInformation'
    missing_slice_interp_method = 'intermediate_nonlin_mean'
    sdf_clip_val = 0
    cortical_detail_weight = None #not used when not SDF

output_dir = f'/tmp/{subject}_sliceReg_optimized_v2_rescale_{rescale}{sdf_tag}/'
_df = pd.read_csv('/data/neuralabc/neuralabc_volunteers/macaque/all_TP_image_idxs_file_lookup.csv')

missing_idxs_to_fill = [32,59,120,160,189,228] #these are the slice indices with missing or terrible data, fill with coreg of neighbours
# output_dir = '/data/data_drive/Macaque_CB/processing/results_from_cell_counts/slice_reg_perSliceTemplate_image_weights_all_tmp/'
## _df = pd.read_csv('/data/data_drive/Macaque_CB/processing/results_from_cell_counts/all_TP_image_idxs_file_lookup.csv')

#missing_idxs_to_fill = [32]
# missing_idxs_to_fill = [5]
# missing_idxs_to_fill = None
all_image_fnames = list(_df['file_name'].values)


print('*********************************************************************************************************')
print(f'Output directory: {output_dir}')
print('*********************************************************************************************************')

# set missing indices, which will be iteratively filled with the mean of the neighbouring slices
if missing_idxs_to_fill is not None:
    if np.max(np.array(missing_idxs_to_fill)) > len(all_image_fnames): #since these are indices, will start @ 0
        raise ValueError("Missing slice indices exceed the number of images in the stack.")

# all_image_fnames = all_image_fnames[0:10] #for testing
all_image_names = [os.path.basename(image).split('.')[0] for image in all_image_fnames] #remove the .tif extension to comply with formatting below


*********************************************************************************************************
Output directory: /tmp/zefir_sliceReg_optimized_v2_rescale_40/
*********************************************************************************************************


In [20]:
all_image_names

['_Image_01_-_20x_01_cellCount_29_downsample_10p002um_pix',
 '_Image_01_-_20x_02_cellCount_29_downsample_10p002um_pix',
 '_Image_02_-_20x_01_cellCount_29_downsample_10p002um_pix',
 '_Image_02_-_20x_02_cellCount_29_downsample_10p002um_pix',
 '_Image_03_-_20x_01_cellCount_29_downsample_10p002um_pix',
 '_Image_03_-_20x_02_cellCount_29_downsample_10p002um_pix',
 '_Image_04_-_20x_01_cellCount_29_downsample_10p002um_pix',
 '_Image_04_-_20x_02_cellCount_29_downsample_10p002um_pix',
 '_Image_05_-_20x_01_cellCount_29_downsample_10p002um_pix',
 '_Image_05_-_20x_02_cellCount_29_downsample_10p002um_pix',
 '_Image_06_-_20x_01_cellCount_29_downsample_10p002um_pix',
 '_Image_06_-_20x_02_cellCount_29_downsample_10p002um_pix',
 '_Image_07_-_20x_01_cellCount_29_downsample_10p002um_pix',
 '_Image_07_-_20x_02_cellCount_29_downsample_10p002um_pix',
 '_Image_08_-_20x_01_cellCount_29_downsample_10p002um_pix',
 '_Image_08_-_20x_02_cellCount_29_downsample_10p002um_pix',
 '_Image_09_-_20x_01_cellCount_29_downsa

In [18]:
import matplotlib.pyplot as plt
import numpy as np
np.max(np.diff(_df['Unnamed: 0'].to_list()))
# _df.head()

147